### Baseline Model Development

Goal:
Build the first baseline model for loan risk prediction using the cleaned dataset.

#### Baseline Model Development

Goal:
Predict whether a borrower will default on a loan.

Target:
is_default
0 = Fully Paid
1 = Charged Off / Default

Current Dataset:
X.shape = (87889, 89)
y.shape = (87889,)

Planned Baseline:
Logistic Regression

Evaluation Strategy:
- Stratified Train/Test Split
- Accuracy not used as primary metric
- Focus on classification metrics suitable for imbalanced data

## Next Steps

- Perform stratified 80/20 train-test split
- Train Logistic Regression baseline
- Evaluate using ROC-AUC, Precision, Recall, F1-score
- Compare with future tree-based models

In [2]:
import pandas as pd

In [3]:
X = pd.read_csv("../data/processed/X.csv", index_col=0)

In [4]:
X.shape

(87889, 90)

In [5]:
y =pd.read_csv("../data/processed/y.csv", index_col=0).squeeze()

In [6]:
y.shape

(87889,)

In [7]:
X.index.equals(y.index)

True

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size=0.2,
    random_state=42, 
    stratify=y
)

In [10]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (70311, 90)
X_test : (17578, 90)
y_train: (70311,)
y_test : (17578,)


In [20]:
print("Training distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Training distribution:
loan_status
0    0.799704
1    0.200296
Name: proportion, dtype: float64

Test distribution:
loan_status
0    0.799693
1    0.200307
Name: proportion, dtype: float64


In [11]:
X.dtypes.value_counts()

float64    68
bool       14
int64       8
Name: count, dtype: int64

In [12]:
X.select_dtypes(include="bool").columns.tolist()


['MORTGAGE',
 'OWN',
 'RENT',
 'credit_card',
 'debt_consolidation',
 'home_improvement',
 'house',
 'major_purchase',
 'medical',
 'moving',
 'other',
 'renewable_energy',
 'small_business',
 'vacation']

In [13]:
X.select_dtypes(include="int64").columns.to_list()

['id',
 'term',
 'sub_grade',
 'verification_status',
 'initial_list_status',
 'application_type',
 'debt_settlement_flag',
 'is_default']

In [14]:
y.value_counts()

loan_status
0    70285
1    17604
Name: count, dtype: int64

In [15]:
X["is_default"].value_counts()

is_default
0    70285
1    17604
Name: count, dtype: int64

In [16]:
X[['term', 'sub_grade', 'verification_status',
   'initial_list_status', 'application_type',
   'debt_settlement_flag']].nunique()

term                     2
sub_grade               35
verification_status      3
initial_list_status      2
application_type         2
debt_settlement_flag     2
dtype: int64

In [17]:
X[['term', 'sub_grade', 'verification_status',
   'initial_list_status', 'application_type',
   'debt_settlement_flag']].head()

,term,sub_grade,verification_status,initial_list_status,application_type,debt_settlement_flag
0,0,13,0,0,1,0
1,0,10,0,0,1,0
2,1,8,0,0,0,0
4,1,25,1,0,1,0
5,0,12,1,0,1,0


In [18]:
X.select_dtypes(include="float64").columns.tolist()

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'int_rate',
 'installment',
 'emp_length',
 'annual_inc',
 'dti',
 'delinq_2yrs',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'out_prncp',
 'out_prncp_inv',
 'recoveries',
 'collection_recovery_fee',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'policy_code',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'total_rev_hi_lim',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 'mths_since_recent_inq',
 'mths_since_recent_revol_delinq',
 'num_accts_ever_120_pd',
 'num_actv_bc_tl',
 'num_actv_rev_tl',
 'num_bc_sats',
 'num_bc_tl',
 'num_il_tl',
 'num_op_rev_tl',
 'num_rev_accts',
 'num_rev_tl_bal

In [19]:
X.select_dtypes(include="float64").nunique().sort_values().to_frame("n_unique")

,n_unique
policy_code,1
out_prncp_inv,2
out_prncp,2
num_tl_120dpd_2m,3
num_tl_30dpd,5
...,...
revol_bal,35372
total_il_high_credit_limit,50710
total_bal_ex_mort,59777
tot_hi_cred_lim,70343


In [20]:
X.select_dtypes(include="float64").nunique().sort_values().head(15)

policy_code                    1
out_prncp_inv                  2
out_prncp                      2
num_tl_120dpd_2m               3
num_tl_30dpd                   5
acc_now_delinq                 5
inq_last_6mths                 6
collections_12_mths_ex_med     6
pub_rec_bankruptcies           8
chargeoff_within_12_mths       8
emp_length                    12
num_tl_90g_dpd_24m            17
delinq_2yrs                   19
tax_liens                     20
pub_rec                       22
dtype: int64

In [21]:
X.select_dtypes(include="float64").nunique().sort_values().head(25)

policy_code                    1
out_prncp_inv                  2
out_prncp                      2
num_tl_120dpd_2m               3
num_tl_30dpd                   5
acc_now_delinq                 5
inq_last_6mths                 6
collections_12_mths_ex_med     6
pub_rec_bankruptcies           8
chargeoff_within_12_mths       8
emp_length                    12
num_tl_90g_dpd_24m            17
delinq_2yrs                   19
tax_liens                     20
pub_rec                       22
mort_acc                      24
mths_since_recent_inq         26
num_tl_op_past_12m            26
num_actv_bc_tl                27
num_accts_ever_120_pd         29
num_rev_tl_bal_gt_0           37
fico_range_low                38
fico_range_high               38
num_bc_sats                   38
num_actv_rev_tl               40
dtype: int64

In [22]:
X_train.dtypes.value_counts()

float64    68
bool       14
int64       8
Name: count, dtype: int64

In [24]:
excluded_cols = [
    "id",
    "is_default",
    "debt_settlement_flag",
    "policy_code",
    "out_prncp",
    "out_prncp_inv",
]

In [25]:
[col for col in [
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
] if col in X.columns]

[]

In [26]:
X_train = X_train.drop(columns=excluded_cols)
X_test = X_test.drop(columns=excluded_cols)

In [27]:
[col for col in excluded_cols if col in X_train.columns or col in X_test.columns]

[]

In [28]:
numerical_cols = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
numerical_cols

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'sub_grade',
 'emp_length',
 'annual_inc',
 'verification_status',
 'dti',
 'delinq_2yrs',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'recoveries',
 'collection_recovery_fee',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'application_type',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'total_rev_hi_lim',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 'mths_since_recent_inq',
 'mths_since_recent_revol_delinq',
 'num_accts_ever_120_pd',
 'num_actv_bc_tl',
 'num_actv_rev_tl',
 'num_bc_sats',
 'num_bc_tl',
 'num_il_tl',
 'num_op

In [32]:
binary_cols = X_train.select_dtypes(include=["bool"]).columns.to_list()
binary_cols

['MORTGAGE',
 'OWN',
 'RENT',
 'credit_card',
 'debt_consolidation',
 'home_improvement',
 'house',
 'major_purchase',
 'medical',
 'moving',
 'other',
 'renewable_energy',
 'small_business',
 'vacation']

In [33]:
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.to_list()
categorical_cols

[]

In [34]:
categorical_cols = ["verification_status"]

In [35]:
"verification_status" in numerical_cols

True

In [36]:
numerical_cols.remove("verification_status")

In [37]:
"verification_status" in numerical_cols

False

In [38]:
low_cardinality = (
    X_train.nunique()
    .sort_values()
    .loc[lambda s: (s >= 3) & (s <= 12)]
)

low_cardinality

verification_status            3
num_tl_120dpd_2m               3
acc_now_delinq                 4
num_tl_30dpd                   4
collections_12_mths_ex_med     6
inq_last_6mths                 6
pub_rec_bankruptcies           7
chargeoff_within_12_mths       8
emp_length                    12
dtype: int64

In [39]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(), categorical_cols),
        ("bin", "passthrough", binary_cols)
    ],
    remainder="drop"
)

In [40]:
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_n

In [41]:
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [42]:
X_train_transformed.shape, X_test_transformed.shape

((70311, 86), (17578, 86))

In [43]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [44]:
model.fit(X_train_transformed, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb